In [ ]:
%pip install libpysal 

In [ ]:
%pip install --force-reinstall "pandas<2.0.0"

In [ ]:
from SEDataObjects import *
from SEDataObjects.BaseLayer import *
from SEDataObjects.BaseLayer.GBFSDataObject import *
from SEDataObjects.transitWrappers import *
from SEDataObjects.OSMStreetlightsDataObject import *
from SEDataObjects.constants import GEODESIC_CRS
from SEDataObjects.BaseLayer.osm_intersection_metrics import (calculate_osm_intersection_metrics,)
import geopandas as gpd
import folium

In [ ]:
CONFIG_MAP_AREA_PATH = "map_area_path"
CONFIG_MAP_AREA_PROJECTED_CRS = "map_area_projected_crs"
CONFIG_GTFS_TRANSITLAND_URL = "gtfs_url"
CONFIG_GTFS_TRANSITLAND_KEY_PATH = "gtfs_key_path"
CONFIG_GTFS_CACHE_FOLDER = "gtfs_cache_folder"
CONFIG_FTA_FACILITY_INVENTORY_PATH = "fta_facility_inventory_path"
CONFIG_CITYBIKES_URL = "citybikes_url"
CONFIG_AFDC_API_KEY = "afdc_api_key"
CONFIG_AFDC_URL = "afdc_url"
CONFIG_CARTO_API_KEY = "carto_api_key"
CONFIG_CARTO_POSITRON_URL = "carto_positron_url"
CONFIG_CARTO_VOYAGER_URL = "carto_voyager_url"
CONFIG_OSM_CACHE_FOLDER = "osm_cache_folder"
CONFIG_EPA_EJSCREEN_PATH = "epa_ejscreen_path"
CONFIG_SMART_LOCATION_PATH = "smart_location_path"
CONFIG_SMART_LOCATION_TRANSIT_PATH = "smart_location_transit_path"
CONFIG_GBFS_URLS = "gbfs_urls"

# COMPLETE CONFIG HERE
CONFIG = {
    CONFIG_MAP_AREA_PATH: "./rawData/San Francisco.geojson", # The path to the map area spatial data file
    CONFIG_MAP_AREA_PROJECTED_CRS: 3310, # The EPSG number for a projected crs with units in meters covering the map area
    CONFIG_GTFS_TRANSITLAND_URL: "https://transit.land/api/v2/rest/feeds.json", # A path to the Transitland V2 api. See https://www.interline.io/transitland/plans-pricing/
    CONFIG_GTFS_TRANSITLAND_KEY_PATH: "./rawData/TRANSITLAND_KEY.txt", # A path to a file contaiing your Transitland API key
    CONFIG_GTFS_CACHE_FOLDER: "./cache/gtfs_cache/SF_city", # A path to a folder where GTFS-static feeds and extracted data canbe stored. The folder must exist, and is recommended to be in cache/gtfs_cache/**
    CONFIG_FTA_FACILITY_INVENTORY_PATH: "./rawData/2023 Facility Inventory.xlsx", # A path to the FTA facility inventory sheet
    CONFIG_CITYBIKES_URL: "http://api.citybik.es/", # The Citybikes API endpoint. See https://citybik.es
    CONFIG_OSM_CACHE_FOLDER: "./cache/osmnx_cache", # A folder to use as the OSMNX cache
    CONFIG_AFDC_URL: "https://developer.nlr.gov/api/alt-fuel-stations/v1/nearest.geojson", # A url to the NRL AFDC alternate fuel stations nearby api endpoint
    CONFIG_AFDC_API_KEY: "./rawData/AFDC_API_KEY.txt", # A path to a folder with a NREL api key. See https://developer.nrel.gov/docs/api-key/
    CONFIG_CARTO_API_KEY: "./rawData/CARTO_API_KEY.txt",
    CONFIG_CARTO_POSITRON_URL: "https://basemaps.cartocdn.com/light_all/{z}/{x}/{y}.png?key=cb1_28hp_1_7d27b08a50e3bcb826624cb7",
    CONFIG_CARTO_VOYAGER_URL: "https://basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}.png?key=cb1_28hp_1_7d27b08a50e3bcb826624cb7",
    CONFIG_EPA_EJSCREEN_PATH: "rawData/EJScreen_2024_BG_with_AS_CNMI_GU_VI.gdb", # A path to the Ejscreen gdb
    CONFIG_SMART_LOCATION_PATH: "rawData/SmartLocationDatabaseV3/SmartLocationDatabase.gdb", # A path to the Smart Location DB gdb
    CONFIG_SMART_LOCATION_TRANSIT_PATH: "rawData/SLD_Trans45/SLD_Trans45.gdb", # A path to the Smart Location DB Transit Tool gdb
    CONFIG_GBFS_URLS: [
        "https://gbfs.lyftbikes.com/gbfs/2.3/gbfs.json",
    ],
}

In [ ]:
from shapely.geometry import Polygon, MultiPolygon

# Load in projected CRS for accurate area
map_area_gdf = gpd.read_file(CONFIG[CONFIG_MAP_AREA_PATH]).to_crs(CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS])

# Dissolve first so we can split into individual components
geom = map_area_gdf.unary_union

# Break MultiPolygon into parts
if isinstance(geom, Polygon):
    parts = [geom]
elif isinstance(geom, MultiPolygon):
    parts = list(geom.geoms)
else:
    parts = [g for g in getattr(geom, "geoms", []) if isinstance(g, Polygon)]

parts_gdf = gpd.GeoDataFrame(geometry=parts, crs=map_area_gdf.crs)
parts_gdf["area_m2"] = parts_gdf.area

N = 2
kept = parts_gdf.nlargest(N, "area_m2").copy()

# Convert back to geodesic CRS if needed by loaders
kept = kept.to_crs(GEODESIC_CRS)

# Final dissolved map area
map_area = kept.unary_union
map_area

In [ ]:
# Instantiate data objects

gtfs_instance = GTFSDataObject(
    CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS],
    CONFIG[CONFIG_GTFS_CACHE_FOLDER],
    CONFIG[CONFIG_GTFS_TRANSITLAND_URL],
    api_key_path=CONFIG[CONFIG_GTFS_TRANSITLAND_KEY_PATH],
    download_transitland_first=True,
    load_from_cache=False             
)

In [ ]:
import pandas as pd

# Overwrite df_feeds_metadata with a customized version that handles dictionary assignment smoothly
original_init = pd.DataFrame.__init__

def patched_init(self, *args, **kwargs):
    original_init(self, *args, **kwargs)
    # Give the DataFrame's .loc a fallback mapping capability for this specific notebook run
    if hasattr(self, 'columns') and 'last_fetch_succeeded' in args or 'columns' in kwargs:
        pass

# Force the instance to ignore the dictionary strictness by patching the specific object instance
if hasattr(gtfs_instance, 'load_data'):
    # Let's bypass the strict dataframe construction line entirely by preemptively creating the dataframe structure it wants
    import datetime as dt
    columns_needed = [
        "name", "agency_url", "url", "raw_feed_path", "last_fetched", 
        "attribution_url", "attribution_text", "attribution_instructions", 
        "attribution_must_attribute", "last_fetch_succeeded"
    ]
    # Inject a cleanly built dataframe directly into the object so it doesn't try to build a broken empty one on line 159
    gtfs_instance.df_feeds_metadata = pd.DataFrame(columns=columns_needed)

Add Census API Key

In [ ]:
import os
import time
import requests
os.environ["CENSUS_API_KEY"] = "3f2744b1ffe0324053bf51aab03594cd2d00ec53"

In [ ]:
import numpy as np
fta_instance = FTAFacilityInventoryDataObject(
    CONFIG[CONFIG_FTA_FACILITY_INVENTORY_PATH],
    CONFIG[CONFIG_OSM_CACHE_FOLDER]
)
gbfs_instance = GBFSDataObject(
    CONFIG[CONFIG_GBFS_URLS]
)
afdc_instance = AFDCApiDataObject(CONFIG[CONFIG_AFDC_URL], CONFIG[CONFIG_AFDC_API_KEY],CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS])
osm_bike_parking_instance = OSMBikeParkingDataObject(CONFIG[CONFIG_OSM_CACHE_FOLDER], {"amenity": ["bicycle_parking"]})
smart_location_info = SmartLocationWrapper(
    CONFIG[CONFIG_SMART_LOCATION_PATH],
    CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS],
    CONFIG[CONFIG_SMART_LOCATION_TRANSIT_PATH],
)

base_layer_instance = BaseLayer(
    [
        SmartLocationPopulationDensity(smart_location_info),
        SmartLocationJobDensity(smart_location_info),
        SmartLocationRetailEntertainmentJobDensity(smart_location_info),
        SmartLocationRawJobs(smart_location_info),
        SmartLocationPctPopAccessibleByTransit(smart_location_info),
        SmartLocationPctJobsByTransit(smart_location_info),
        SmartLocationPctWorkersByTransit(smart_location_info),
        SmartLocationPctLowWageWorkersByTransit(smart_location_info),
        SmartLocationPctLowMedWageWorkersByTransit(smart_location_info),
        CensusModeshare(),
        CensusCarlessness(),
        SmartLocationNationalWalkabilityIndex(smart_location_info),
        BaseLayerEjscreen(CONFIG[CONFIG_EPA_EJSCREEN_PATH]),
    ],
    CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS],
    ColorMaps.SMART_LOCATION_PCT_POP_ACCESSIBLE_BY_TRANSIT_COLORMAP,
    smooth=True,
    remove_water=True # This should be set to False if TIGER goes down
)
bike_instance = OSMBikeStreetsDataObject(
    CONFIG[CONFIG_OSM_CACHE_FOLDER],
    reference=gtfs_instance,
    local_crs=CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS]
)
streetlights_instance = OSMStreetlightsDataObject(
    CONFIG[CONFIG_OSM_CACHE_FOLDER]
)

# Edit these columns to change the objects displayed on the map
all_objects = {
    "GTFS": gtfs_instance,
    "BASE": base_layer_instance,
    "BIKE": bike_instance,
    "FTA": fta_instance,
    "GBFS": gbfs_instance,
    "AFDC": afdc_instance,
    "OSM BIKE PARKING": osm_bike_parking_instance,
    "STREETLIGHTS": streetlights_instance
}

objects_load_order = (
    "BASE",
    "BIKE",
    "GTFS",
    "FTA",
    "AFDC",
    "OSM BIKE PARKING",
    "GBFS",
    "STREETLIGHTS"
)

# This should only contain GTFS
objects_must_await = (
    "GTFS"
)

import requests

def debug_json(url, params=None, headers=None, timeout=30):
    r = requests.get(url, params=params, headers=headers, timeout=timeout)
    print("URL:", r.url)
    print("STATUS:", r.status_code)
    print("CONTENT-TYPE:", r.headers.get("Content-Type"))
    print("FIRST 500 CHARS:\n", r.text[:500])
    try:
        data = r.json()
        print("JSON OK. Top-level type:", type(data))
        return data
    except Exception as e:
        print("JSON parse failed:", repr(e))
        return None
    
await gtfs_instance.load_data(map_area, GEODESIC_CRS)
base_layer_instance.load_data(map_area, GEODESIC_CRS)

gdf = calculate_osm_intersection_metrics(
    base_gdf=base_layer_instance.gdf,
    map_area=map_area,
    map_area_crs=GEODESIC_CRS,
    projected_crs=CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS],
    cache_folder=CONFIG[CONFIG_OSM_CACHE_FOLDER],
)

base_layer_instance._gdf = gdf

In [ ]:
print(
    gdf[
        [
            "Intersection Count",
            "Road Centerline Miles",
            "Road Lane Miles",
            "Intersections per Centerline Mile",
            "Intersections per Lane Mile",
        ]
    ].head()
)

#### Mobility Hub Map

In [ ]:
# Mobility Hub Only Map (with CARTO key)
mobility_hub_instance = MobilityHubDataObject(
    gtfs_instance, base_layer_instance, CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS]
)
mobility_hub_instance.load_data(map_area, GEODESIC_CRS)
carto_voyager_tiles = CONFIG[CONFIG_CARTO_VOYAGER_URL]
mobility_hub_map_old = folium.Map(
    location=(map_area.centroid.y, map_area.centroid.x),
    zoom_start=12,
    tiles=None,              # IMPORTANT: don't use shorthand
    prefer_canvas=True
)

folium.TileLayer(
    tiles=carto_voyager_tiles,
    name="CARTO Voyager",
    attr='&copy; OpenStreetMap contributors &copy; CARTO',
    overlay=False,
    control=True,
    max_zoom=20
).add_to(mobility_hub_map_old)

base_layer_instance.get_folium_plot().add_to(mobility_hub_map_old)
mobility_hub_instance.get_folium_plot().add_to(mobility_hub_map_old)

mobility_hub_map_old.save("mobility_hub_map_old.html")

In [ ]:
mobility_hub_instance = MobilityHubDataObject(gtfs_instance, base_layer_instance, CONFIG[CONFIG_MAP_AREA_PROJECTED_CRS])
mobility_hub_instance.load_data(map_area, GEODESIC_CRS)

# 1. Initialize the map canvas with detailed street grids and water bodies
mobility_hub_map1 = folium.Map(
    location=(map_area.centroid.y, map_area.centroid.x),
    tiles="CartoDB Voyager",  
    zoom_start=13,            
    prefer_canvas=True
)
carto_voyager_tiles = CONFIG[CONFIG_CARTO_VOYAGER_URL]
mobility_hub_map1 = folium.Map(
    location=(map_area.centroid.y, map_area.centroid.x),
    zoom_start=12,
    tiles=None,              # IMPORTANT: don't use shorthand
    prefer_canvas=True
)

folium.TileLayer(
    tiles=carto_voyager_tiles,
    name="CARTO Voyager",
    attr='&copy; OpenStreetMap contributors &copy; CARTO',
    overlay=False,
    control=True,
    max_zoom=20
).add_to(mobility_hub_map1)
# 2. Add your underlying demographic walkability/density layer
base_layer_instance.get_folium_plot().add_to(mobility_hub_map1)

# 3. Extract transit data and convert to standard map coordinates
hubs_gdf = mobility_hub_instance._gdf.to_crs(epsg=4326)

# 4. Loop through the hubs and isolate the classifications
for idx, row in hubs_gdf.iterrows():
    tb_val = str(row['trunk_branch_type']).lower().split('.')[-1]
    mode_val = str(row['mode']).lower().split('.')[-1]
    
    # Skip plotting points that are explicitly flagged as NOT being mobility hubs
    if "not_mobility_hub" in tb_val or "not_relevant" in tb_val:
        continue

    # Default styling definitions (Fallback)
    color = "#7f8c8d"  # Grey
    radius = 5
    
    # Fine-grained Mode and Hierarchy Differentiation
    if "cable_car" in mode_val or "cablecar" in mode_val:
        if "trunk" == tb_val:
            color = "#e31700"  # Bright Crimson Red for SF Cable Cars
            radius = 8
        elif "branch" == tb_val:
            color = "#ea7164"  # Light Red for SF Cable Cars
            radius = 5
    elif "tram" in mode_val or "streetcar" in mode_val or "light_rail" in mode_val:
        if "trunk" == tb_val:
            color = "#00491e"  # Dark Green for Trams / Muni Metro Streetcars
            radius = 8
        elif "branch" == tb_val:
            color = "#2ecc71"  # Green for Trams / Muni Metro Streetcars
            radius = 6
    elif "rail" in mode_val or "subway" in mode_val or "train" in mode_val:
        if "trunk" == tb_val:
            color = "#0dff00"  # Light Lime Green for regional rail (BART / Caltrain)
            radius = 8
        elif "branch" == tb_val:
            color = "#2980b9"  # Deep Blue for Branch Rail lines
            radius = 6
    elif "bus" in mode_val:
        if "trunk" == tb_val:
            color = "#f49eff"  # Light Pink for heavy-frequency Trunk Bus corridors
            radius = 8
        elif "branch" == tb_val:
            color = "#d9d5db"  # Light Gray for Branch Bus lines
            radius = 6
        else:
            color = "#f1c40f"  # Yellow for Local Bus links
            radius = 4

    # --- INDENTED INSIDE THE LOOP: Construct HTML Popup Data Table ---
    od_type = str(row.get('od_type', 'N/A')).split('.')[-1].capitalize()
    od_score = f"{float(row.get('od_score', 0)):.3f}" if row.get('od_score') is not None else "N/A"
    tb_type = str(row.get('trunk_branch_type', 'N/A')).split('.')[-1].capitalize()
    mode_display = str(row.get('mode', 'N/A')).split('.')[-1].capitalize()
    headway = f"{float(row.get('adjusted_headway', 0)):.1f}" if row.get('adjusted_headway') is not None else "N/A"
    freq = f"{float(row.get('total_frequency', 0)):.3f}" if row.get('total_frequency') is not None else "N/A"
    transfer = str(row.get('transfer', 'False')).lower()

    popup_html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 12px; line-height: 1.4; min-width: 220px;">
        <table style="width: 100%; border-collapse: collapse;">
            <tr style="border-bottom: 1px solid #eee; background-color: #f9f9f9;"><td style="padding: 4px; font-weight: bold; color: #555;">Origin / Local?</td><td style="padding: 4px; text-align: right;">{od_type}</td></tr>
            <tr style="border-bottom: 1px solid #eee;"><td style="padding: 4px; font-weight: bold; color: #555;">Origin / Local Score</td><td style="padding: 4px; text-align: right;">{od_score}</td></tr>
            <tr style="border-bottom: 1px solid #eee; background-color: #f9f9f9;"><td style="padding: 4px; font-weight: bold; color: #555;">Trunk / Branch?</td><td style="padding: 4px; text-align: right; font-weight: bold; color: {'#d32f2f' if 'Trunk' in tb_type else '#e67e22'};">{tb_type}</td></tr>
            <tr style="border-bottom: 1px solid #eee;"><td style="padding: 4px; font-weight: bold; color: #555;">Mode</td><td style="padding: 4px; text-align: right;">{mode_display}</td></tr>
            <tr style="border-bottom: 1px solid #eee; background-color: #f9f9f9;"><td style="padding: 4px; font-weight: bold; color: #555;">Minimum Headway (one direction)</td><td style="padding: 4px; text-align: right;">{headway}</td></tr>
            <tr style="border-bottom: 1px solid #eee;"><td style="padding: 4px; font-weight: bold; color: #555;">Total Frequency (all directions)</td><td style="padding: 4px; text-align: right;">{freq}</td></tr>
            <tr style="background-color: #f9f9f9;"><td style="padding: 4px; font-weight: bold; color: #555;">Transfer</td><td style="padding: 4px; text-align: right; color: {'green' if transfer == 'true' else 'red'};">{transfer}</td></tr>
        </table>
    </div>
    """

    # 5. Draw solid, high-contrast markers onto the canvas (Passing the custom HTML)
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=radius,
        color="#ffffff",       
        weight=1.2,            
        fill=True,
        fill_color=color,      
        fill_opacity=0.9,      
        popup=folium.Popup(popup_html, max_width=350)
    ).add_to(mobility_hub_map1)

# 6. Save the output and show map (original finish, completely outside the loop)
mobility_hub_map1.save("mobility_hub_map1.html")
mobility_hub_map1

In [ ]:
import branca.colormap as cm
from SEDataObjects.BaseLayer.constants import (
    SMART_LOCATION_PCT_POP_ACCESSIBLE_BY_TRANSIT_NAME,
    SMART_LOCATION_PCT_JOBS_BY_TRANSIT_NAME,
    SMART_LOCATION_PCT_WORKERS_BY_TRANSIT_NAME,
    SMART_LOCATION_PCT_LOW_WAGE_WORKERS_BY_TRANSIT_NAME,
    SMART_LOCATION_PCT_LOW_MED_WAGE_WORKERS_BY_TRANSIT_NAME,
)

# Ensure numeric for target columns
metric_names = [
    SMART_LOCATION_PCT_POP_ACCESSIBLE_BY_TRANSIT_NAME,
    SMART_LOCATION_PCT_JOBS_BY_TRANSIT_NAME,
    SMART_LOCATION_PCT_WORKERS_BY_TRANSIT_NAME,
    SMART_LOCATION_PCT_LOW_WAGE_WORKERS_BY_TRANSIT_NAME,
    SMART_LOCATION_PCT_LOW_MED_WAGE_WORKERS_BY_TRANSIT_NAME,
]
for c in metric_names:
    if c in gdf.columns:
        gdf[c] = pd.to_numeric(gdf[c], errors="coerce")

pct_cols = [
    "Percentage of Population with Access by Transit",
    "Percentage of Regional Jobs Accessible by Transit",
    "Percentage of Workers with Access by Transit",
    "Percentage of All Low-Wage Workers with Access by Transit",
    "Percentage of Low-Medium Wage Workers with Access by Transit",
]

for c in pct_cols:
    if c in gdf.columns:
        gdf[c] = pd.to_numeric(gdf[c], errors="coerce")
        gdf[f"{c} (%)"] = gdf[c].map(lambda v: f"{v*100:.1f}%" if pd.notnull(v) else "N/A")

ACRE_TO_KM2_FACTOR = 247.105381

density_cols = [
    "Population Density (people/acre)",
    "Job Density (jobs/acre)",
    "Retail and Entertainment Job Density (jobs/acre)",
]

for c in density_cols:
    if c in gdf.columns:
        gdf[c] = pd.to_numeric(gdf[c], errors="coerce")
        gdf[c.replace("/acre", "/km²")] = gdf[c].map(
            lambda v: f"{v * ACRE_TO_KM2_FACTOR:,.1f}" if pd.notnull(v) else "N/A"
        )

# Building Map
m = folium.Map(
    location=(map_area.centroid.y, map_area.centroid.x),
    zoom_start=12,
    tiles=None,
    prefer_canvas=True
)

# Use keyed CARTO URL from config
carto_voyager_tiles = CONFIG[CONFIG_CARTO_VOYAGER_URL]

folium.TileLayer(
    tiles=carto_voyager_tiles,
    name="CARTO Voyager",
    attr='&copy; OpenStreetMap contributors &copy; CARTO',
    overlay=False,
    control=True,
    max_zoom=20
).add_to(m)
hubs_layer = folium.FeatureGroup(name="Mobility Hubs (mode/trunk colors)", show=True)
hubs_gdf = mobility_hub_instance._gdf.to_crs(epsg=4326)

for _, row in hubs_gdf.iterrows():
    tb_val = str(row['trunk_branch_type']).lower().split('.')[-1]
    mode_val = str(row['mode']).lower().split('.')[-1]

    if "not_mobility_hub" in tb_val or "not_relevant" in tb_val:
        continue

    color = "#7f8c8d"
    radius = 5

    if "cable_car" in mode_val or "cablecar" in mode_val:
        color, radius = ("#e31700", 8) if tb_val == "trunk" else ("#ea7164", 5)
    elif "tram" in mode_val or "streetcar" in mode_val or "light_rail" in mode_val:
        color, radius = ("#00491e", 8) if tb_val == "trunk" else ("#2ecc71", 6)
    elif "rail" in mode_val or "subway" in mode_val or "train" in mode_val:
        color, radius = ("#0dff00", 8) if tb_val == "trunk" else ("#2980b9", 6)
    elif "bus" in mode_val:
        if tb_val == "trunk":
            color, radius = "#f49eff", 8
        elif tb_val == "branch":
            color, radius = "#d9d5db", 6
        else:
            color, radius = "#f1c40f", 4

    od_type = str(row.get('od_type', 'N/A')).split('.')[-1].capitalize()
    od_score = f"{float(row.get('od_score', 0)):.3f}" if row.get('od_score') is not None else "N/A"
    tb_type = str(row.get('trunk_branch_type', 'N/A')).split('.')[-1].capitalize()
    mode_display = str(row.get('mode', 'N/A')).split('.')[-1].capitalize()
    headway = f"{float(row.get('adjusted_headway', 0)):.1f}" if row.get('adjusted_headway') is not None else "N/A"
    freq = f"{float(row.get('total_frequency', 0)):.3f}" if row.get('total_frequency') is not None else "N/A"
    transfer = str(row.get('transfer', 'False')).lower()

    popup_html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 12px; line-height: 1.4; min-width: 220px;">
        <table style="width: 100%; border-collapse: collapse;">
            <tr style="border-bottom: 1px solid #eee; background-color: #f9f9f9;"><td style="padding: 4px; font-weight: bold; color: #555;">Origin / Local?</td><td style="padding: 4px; text-align: right;">{od_type}</td></tr>
            <tr style="border-bottom: 1px solid #eee;"><td style="padding: 4px; font-weight: bold; color: #555;">Origin / Local Score</td><td style="padding: 4px; text-align: right;">{od_score}</td></tr>
            <tr style="border-bottom: 1px solid #eee; background-color: #f9f9f9;"><td style="padding: 4px; font-weight: bold; color: #555;">Trunk / Branch?</td><td style="padding: 4px; text-align: right; font-weight: bold; color: {'#d32f2f' if 'Trunk' in tb_type else '#e67e22'};">{tb_type}</td></tr>
            <tr style="border-bottom: 1px solid #eee;"><td style="padding: 4px; font-weight: bold; color: #555;">Mode</td><td style="padding: 4px; text-align: right;">{mode_display}</td></tr>
            <tr style="border-bottom: 1px solid #eee; background-color: #f9f9f9;"><td style="padding: 4px; font-weight: bold; color: #555;">Minimum Headway (one direction)</td><td style="padding: 4px; text-align: right;">{headway}</td></tr>
            <tr style="border-bottom: 1px solid #eee;"><td style="padding: 4px; font-weight: bold; color: #555;">Total Frequency (all directions)</td><td style="padding: 4px; text-align: right;">{freq}</td></tr>
            <tr style="background-color: #f9f9f9;"><td style="padding: 4px; font-weight: bold; color: #555;">Transfer</td><td style="padding: 4px; text-align: right; color: {'green' if transfer == 'true' else 'red'};">{transfer}</td></tr>
        </table>
    </div>
    """

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=radius,
        color="#ffffff",
        weight=1.2,
        fill=True,
        fill_color=color,
        fill_opacity=0.9,
        popup=folium.Popup(popup_html, max_width=350),
    ).add_to(hubs_layer)

hubs_layer.add_to(m)

# Color ramp
ramp_colors = [
    "#ffffb2",  # < 2.5%
    "#fed976",  # 2.5% - 5%
    "#feb24c",  # 5% - 10%
    "#fd8d3c",  # 10% - 20%
    "#fc4e2a",  # 20% - 30%
    "#e31a1c",  # 30% - 50%
    "#b10026",  # > 50%
]

# These are the actual dataframe column names and display names
base_metrics = [
    (
        SMART_LOCATION_PCT_POP_ACCESSIBLE_BY_TRANSIT_NAME,
        "Pct Population with Access by Transit",
    ),
    (
        SMART_LOCATION_PCT_JOBS_BY_TRANSIT_NAME,
        "Pct Regional Jobs with Access by Transit",
    ),
    (
        SMART_LOCATION_PCT_WORKERS_BY_TRANSIT_NAME,
        "Pct Workers with Access by Transit",
    ),
    (
        SMART_LOCATION_PCT_LOW_WAGE_WORKERS_BY_TRANSIT_NAME,
        "Pct Low-Wage Workers with Access by Transit",
    ),
    (
        SMART_LOCATION_PCT_LOW_MED_WAGE_WORKERS_BY_TRANSIT_NAME,
        "Pct Low-Med Wage Workers with Access by Transit",
    ),
]

popup_fields = [
    "Population Density (people/km²)",
    "Job Density (jobs/km²)",
    "Retail and Entertainment Job Density (jobs/km²)",
    "Total # Jobs",
    "Percentage of Population with Access by Transit (%)",
    "Percentage of Regional Jobs Accessible by Transit (%)",
    "Percentage of Workers with Access by Transit (%)",
    "Percentage of All Low-Wage Workers with Access by Transit (%)",
    "Percentage of Low-Medium Wage Workers with Access by Transit (%)",
    "Sustainable Mobility TTW Modeshare",
    "Carlessness Rate",
    "National Walkability Index",
    "National Traffic Proximity & Volume Percentile",
]

popup_fields = [
    field for field in popup_fields
    if field in gdf.columns
]
popup_aliases = popup_fields.copy()


def add_metric_layer(metric_col: str, layer_name: str, show: bool):
    if metric_col not in gdf.columns:
        print(f"Skipping missing metric: {metric_col}")
        return

    # Fixed thresholds expressed as proportions:
    # 2.5% = 0.025, 5% = 0.05, etc.
    percentage_bins = [
        0.0,
        0.025,
        0.05,
        0.10,
        0.20,
        0.30,
        0.50,
        1.0,
    ]

    cmap = cm.StepColormap(
        colors=ramp_colors,
        index=percentage_bins,
        vmin=0.0,
        vmax=1.0,
        caption=layer_name,
    )

    feature_group = folium.FeatureGroup(
        name=layer_name,
        overlay=True,
        show=show,
    )

    def get_metric_color(
        feature,
        metric_column=metric_col,
        color_map=cmap,
    ):
        value = feature["properties"].get(metric_column)

        if value is None:
            return "black"

        try:
            value = float(value)
        except (TypeError, ValueError):
            return "black"

        if not np.isfinite(value):
            return "black"

        # Keep values within the map's expected range.
        value = max(0.0, min(value, 1.0))

        return color_map(value)

    folium.GeoJson(
        gdf.to_json(),
        style_function=lambda feature, color_function=get_metric_color: {
            "fillColor": color_function(feature),
            "color": "#444444",
            "weight": 0.4,
            "fillOpacity": 0.75,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=popup_fields,
            aliases=popup_aliases,
            localize=True,
            sticky=False,
            labels=True,
        ),
        popup=folium.GeoJsonPopup(
            fields=popup_fields,
            aliases=popup_aliases,
            localize=True,
            labels=True,
        ),
    ).add_to(feature_group)

    feature_group.add_to(m)


# Add the first layer as visible and the others as hidden.
for index, (metric_col, layer_name) in enumerate(base_metrics):
    add_metric_layer(
        metric_col=metric_col,
        layer_name=layer_name,
        show=(index == 0),
    )

base_legend = cm.LinearColormap(
    colors=["#ffffb2", "#fed976", "#feb24c", "#fd8d3c", "#fc4e2a", "#e31a1c", "#b10026"],
    index=[0.0, 0.025, 0.05, 0.1, 0.2, 0.3, 0.5],
    vmin=0.0,
    vmax=0.5,
    caption="Base Layer Percentage (selected metric)"
)
#base_legend.add_to(m)

# Remove or comment this:
# base_legend.add_to(m)

combined_legend_html = """
<div style="
position: fixed;
bottom: 40px;
left: 40px;
z-index: 9999;
background-color: white;
border: 2px solid #666;
border-radius: 8px;
padding: 12px 14px;
font-size: 12px;
line-height: 1.4;
box-shadow: 0 2px 6px rgba(0,0,0,0.25);
max-width: 320px;
font-family: Arial, sans-serif;
">
  <div style="font-weight: bold; font-size: 14px; margin-bottom: 8px;">
    Accessiblity Measures
  </div>
  <div style="margin-bottom: 2px;">Base Layer Percentage</div>

  <!-- Base layer bins -->
  <div style="display:flex; align-items:center; margin:3px 0;">
    <span style="width:14px;height:14px;background:#ffffb2;display:inline-block;margin-right:8px;border:1px solid #ddd;"></span>
    less than 2.5%
  </div>
  <div style="display:flex; align-items:center; margin:3px 0;">
    <span style="width:14px;height:14px;background:#fed976;display:inline-block;margin-right:8px;border:1px solid #ddd;"></span>
    2.5% - 5%
  </div>
  <div style="display:flex; align-items:center; margin:3px 0;">
    <span style="width:14px;height:14px;background:#feb24c;display:inline-block;margin-right:8px;border:1px solid #ddd;"></span>
    5% - 10%
  </div>
  <div style="display:flex; align-items:center; margin:3px 0;">
    <span style="width:14px;height:14px;background:#fd8d3c;display:inline-block;margin-right:8px;border:1px solid #ddd;"></span>
    10% - 20%
  </div>
  <div style="display:flex; align-items:center; margin:3px 0;">
    <span style="width:14px;height:14px;background:#fc4e2a;display:inline-block;margin-right:8px;border:1px solid #ddd;"></span>
    20% - 30%
  </div>
  <div style="display:flex; align-items:center; margin:3px 0;">
    <span style="width:14px;height:14px;background:#e31a1c;display:inline-block;margin-right:8px;border:1px solid #ddd;"></span>
    30% - 50%
  </div>
  <div style="display:flex; align-items:center; margin:3px 0 10px 0;">
    <span style="width:14px;height:14px;background:#b10026;display:inline-block;margin-right:8px;border:1px solid #ddd;"></span>
    Greater than 50%
  </div>

  <div style="border-top:1px solid #ddd; margin:8px 0;"></div>
  <div style="font-weight: bold; margin-bottom: 6px;">Mobility Hubs</div>

  <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#e31700;margin-right:6px;"></span>Cable Car (Trunk)</div>
  <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#ea7164;margin-right:6px;"></span>Cable Car (Branch)</div>
  <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#00491e;margin-right:6px;"></span>Tram/Streetcar/Light Rail (Trunk)</div>
  <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#2ecc71;margin-right:6px;"></span>Tram/Streetcar/Light Rail (Branch)</div>
  <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#0dff00;margin-right:6px;"></span>Rail/Subway/Train (Trunk)</div>
  <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#2980b9;margin-right:6px;"></span>Rail/Subway/Train (Branch)</div>
  <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#f49eff;margin-right:6px;"></span>Bus (Trunk)</div>
  <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#d9d5db;margin-right:6px;"></span>Bus (Branch)</div>
  <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#f1c40f;margin-right:6px;"></span>Bus (Local)</div>
  <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#7f8c8d;margin-right:6px;"></span>Other/Default</div>
</div>
"""
m.get_root().html.add_child(folium.Element(combined_legend_html))

folium.LayerControl(collapsed=False).add_to(m)
m.save("mobility_hub_map_layers.html")
m

#### All Elements

In [ ]:
# Load data objects (non-skipping loading)
for name, object in all_objects.items():
    print(name)
    if not object.get_is_loaded():
        if name in objects_must_await:
            await object.load_data(map_area, GEODESIC_CRS)
        else:
            object.load_data(map_area, GEODESIC_CRS)

In [ ]:
# Load data objects with skipping timeout and connection errors
for name, obj in all_objects.items():
    print(name)
    if obj.get_is_loaded():
        continue

    try:
        if name in objects_must_await:
            await obj.load_data(map_area, GEODESIC_CRS)
        else:
            obj.load_data(map_area, GEODESIC_CRS)

    except requests.exceptions.Timeout as e:
        print(f"WARN: Timeout while loading {name}: {e}. Skipping this layer.")
        continue

    except requests.exceptions.ConnectionError as e:
        print(f"WARN: Connection error while loading {name}: {e}. Skipping this layer.")
        continue

    except Exception as e:
        print(f"WARN: Unexpected error while loading {name}: {e}. Skipping this layer.")
        continue

In [ ]:
# Final Map
import json

m = folium.Map(
    location=(map_area.centroid.y, map_area.centroid.x),
    zoom_start=12,
    tiles=None,
    prefer_canvas=True
)

# Use keyed CARTO URL from config
carto_voyager_tiles = CONFIG[CONFIG_CARTO_VOYAGER_URL]

folium.TileLayer(
    tiles=carto_voyager_tiles,
    name="CARTO Voyager",
    attr='&copy; OpenStreetMap contributors &copy; CARTO',
    overlay=False,
    control=True,
    max_zoom=20
).add_to(m)

gdf = gdf.copy()

pct_cols = [
    "Percentage of Population with Access by Transit",
    "Percentage of Regional Jobs Accessible by Transit",
    "Percentage of Workers with Access by Transit",
    "Percentage of All Low-Wage Workers with Access by Transit",
    "Percentage of Low-Medium Wage Workers with Access by Transit",
]
for c in pct_cols:
    if c in gdf.columns:
        gdf[c] = pd.to_numeric(gdf[c], errors="coerce")
        gdf[f"{c} (%)"] = gdf[c].map(lambda v: f"{v*100:.1f}%" if pd.notnull(v) else "N/A")

ACRE_TO_KM2_FACTOR = 247.105381
density_cols = [
    "Population Density (people/acre)",
    "Job Density (jobs/acre)",
    "Retail and Entertainment Job Density (jobs/acre)",
]
for c in density_cols:
    if c in gdf.columns:
        gdf[c] = pd.to_numeric(gdf[c], errors="coerce")
        gdf[c.replace("/acre", "/km²")] = gdf[c].map(
            lambda v: f"{v * ACRE_TO_KM2_FACTOR:,.1f}" if pd.notnull(v) else "N/A"
        )

intersection_numeric_cols = [
    "Intersection Count",
    "Road Centerline Miles",
    "Road Lane Miles",
    "Intersections per Centerline Mile",
    "Intersections per Lane Mile",
]

for column in intersection_numeric_cols:
    if column in gdf.columns:
        gdf[column] = pd.to_numeric(
            gdf[column],
            errors="coerce",
        )

desired_popup_fields = [
    "Population Density (people/km²)",
    "Job Density (jobs/km²)",
    "Retail and Entertainment Job Density (jobs/km²)",
    "Total # Jobs",

    "Intersection Count",
    "Intersections per Centerline Mile",
    "Intersections per Lane Mile",

    "Percentage of Population with Access by Transit (%)",
    "Percentage of Regional Jobs Accessible by Transit (%)",
    "Percentage of Workers with Access by Transit (%)",
    "Percentage of All Low-Wage Workers with Access by Transit (%)",
    "Percentage of Low-Medium Wage Workers with Access by Transit (%)",

    "Sustainable Mobility TTW Modeshare",
    "Carlessness Rate",
    "National Walkability Index",
    "National Traffic Proximity & Volume Percentile",
]

popup_fields = [
    field
    for field in desired_popup_fields
    if field in gdf.columns
]

popup_aliases = popup_fields.copy()

print("Popup fields being used:", popup_fields)

base_metrics = [
    (SMART_LOCATION_PCT_POP_ACCESSIBLE_BY_TRANSIT_NAME, "Pct Population with Access by Transit"),
    (SMART_LOCATION_PCT_JOBS_BY_TRANSIT_NAME, "Pct Regional Jobs with Access by Transit"),
    (SMART_LOCATION_PCT_WORKERS_BY_TRANSIT_NAME, "Pct Workers with Access by Transit"),
    (SMART_LOCATION_PCT_LOW_WAGE_WORKERS_BY_TRANSIT_NAME, "Pct Low-Wage Workers with Access by Transit"),
    (SMART_LOCATION_PCT_LOW_MED_WAGE_WORKERS_BY_TRANSIT_NAME, "Pct Low-Med Wage Workers with Access by Transit"),
]

ramp_colors = ["#ffffb2", "#fed976", "#feb24c", "#fd8d3c", "#fc4e2a", "#e31a1c", "#b10026"]

first = True
for metric_col, layer_name in base_metrics:
    if metric_col not in gdf.columns:
        continue

    s = pd.to_numeric(gdf[metric_col], errors="coerce")
    percentage_bins = [
    0.0,
    0.025,
    0.05,
    0.10,
    0.20,
    0.30,
    0.50,
]

first = True

for metric_col, layer_name in base_metrics:
    if metric_col not in gdf.columns:
        print(f"Skipping missing metric: {metric_col}")
        continue

    cmap = cm.StepColormap(
        colors=ramp_colors,
        index=percentage_bins,
        vmin=0.0,
        vmax=0.50,
        caption=layer_name,
    )

    fg = folium.FeatureGroup(
        name=layer_name,
        overlay=True,
        show=first,
    )
    first = False

    def get_metric_color(
        feature,
        metric_column=metric_col,
        color_map=cmap,
    ):
        value = feature["properties"].get(metric_column)

        if value is None:
            return "black"

        try:
            value = float(value)

            if not np.isfinite(value):
                return "black"

            return color_map(value)

        except (TypeError, ValueError):
            return "black"

    folium.GeoJson(
        gdf.to_json(),
        style_function=lambda feat, color_function=get_metric_color: {
            "fillColor": color_function(feat),
            "color": "#444444",
            "weight": 0.4,
            "fillOpacity": 0.75,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=popup_fields,
            aliases=popup_aliases,
            localize=True,
            sticky=False,
            labels=True,
        ),
        popup=folium.GeoJsonPopup(
            fields=popup_fields,
            aliases=popup_aliases,
            localize=True,
            labels=True,
        ),
    ).add_to(fg)

    fg.add_to(m)

display_names = {
    "BIKE": "Bike Lanes",
    "FTA": "Agency Parking Lots",
    "AFDC": "Public EV Charging Stations",
    "OSM BIKE PARKING": "Bike Parking",
    "GBFS": "Bikeshare Stations",
    "STREETLIGHTS": "Streetlights",
}

for obj_name in ["BIKE", "FTA", "AFDC", "OSM BIKE PARKING", "GBFS", "STREETLIGHTS"]:
    obj = all_objects[obj_name]
    if obj.get_is_loaded() and len(obj.gdf) > 0:
        overlay_fg = folium.FeatureGroup(name=display_names.get(obj_name, obj_name), overlay=True, show=False)
        obj.get_folium_plot().add_to(overlay_fg)
        overlay_fg.add_to(m)

# Mobility Hubs overlay
hubs_layer = folium.FeatureGroup(name="Mobility Hubs", overlay=True, show=True)

hubs_gdf = mobility_hub_instance._gdf.to_crs(epsg=4326)
geom_col = hubs_gdf.geometry.name

for _, row in hubs_gdf.iterrows():
    geom = row[geom_col]
    if geom is None or geom.is_empty or geom.geom_type != "Point":
        continue

    tb_val = str(row.get("trunk_branch_type", "")).lower().split(".")[-1]
    mode_val = str(row.get("mode", "")).lower().split(".")[-1]

    if "not_mobility_hub" in tb_val or "not_relevant" in tb_val:
        continue

    # default fallback
    color = "#7f8c8d"
    radius = 6

    if "cable_car" in mode_val or "cablecar" in mode_val:
        if tb_val == "trunk":
            color, radius = "#e31700", 8
        elif tb_val == "branch":
            color, radius = "#ea7164", 6
    elif "tram" in mode_val or "streetcar" in mode_val or "light_rail" in mode_val or "metro" in mode_val:
        if tb_val == "trunk":
            color, radius = "#00491e", 8
        elif tb_val == "branch":
            color, radius = "#2ecc71", 6
    elif "rail" in mode_val or "subway" in mode_val or "train" in mode_val:
        if tb_val == "trunk":
            color, radius = "#0dff00", 8
        elif tb_val == "branch":
            color, radius = "#2980b9", 6
    elif "bus" in mode_val:
        if tb_val == "trunk":
            color, radius = "#f49eff", 8
        elif tb_val == "branch":
            color, radius = "#d9d5db", 6
        else:
            color, radius = "#f1c40f", 4
    elif "ferry" in mode_val:
        if tb_val == "trunk":
            color, radius = "#1f77b4", 8
        elif tb_val == "branch":
            color, radius = "#aec7e8", 6

    # popup content
    od_type = str(row.get('od_type', 'N/A')).split('.')[-1].capitalize()
    od_score = f"{float(row.get('od_score', 0)):.3f}" if row.get('od_score') is not None else "N/A"
    tb_type = str(row.get('trunk_branch_type', 'N/A')).split('.')[-1].capitalize()
    mode_display = str(row.get('mode', 'N/A')).split('.')[-1].capitalize()
    headway = f"{float(row.get('adjusted_headway', 0)):.1f}" if row.get('adjusted_headway') is not None else "N/A"
    freq = f"{float(row.get('total_frequency', 0)):.3f}" if row.get('total_frequency') is not None else "N/A"
    transfer = str(row.get('transfer', 'False')).lower()

    popup_html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 12px; line-height: 1.4; min-width: 220px;">
        <table style="width: 100%; border-collapse: collapse;">
            <tr style="border-bottom: 1px solid #eee; background-color: #f9f9f9;"><td style="padding: 4px; font-weight: bold; color: #555;">Origin / Local?</td><td style="padding: 4px; text-align: right;">{od_type}</td></tr>
            <tr style="border-bottom: 1px solid #eee;"><td style="padding: 4px; font-weight: bold; color: #555;">Origin / Local Score</td><td style="padding: 4px; text-align: right;">{od_score}</td></tr>
            <tr style="border-bottom: 1px solid #eee; background-color: #f9f9f9;"><td style="padding: 4px; font-weight: bold; color: #555;">Trunk / Branch?</td><td style="padding: 4px; text-align: right; font-weight: bold; color: {'#d32f2f' if 'Trunk' in tb_type else '#e67e22'};">{tb_type}</td></tr>
            <tr style="border-bottom: 1px solid #eee;"><td style="padding: 4px; font-weight: bold; color: #555;">Mode</td><td style="padding: 4px; text-align: right;">{mode_display}</td></tr>
            <tr style="border-bottom: 1px solid #eee; background-color: #f9f9f9;"><td style="padding: 4px; font-weight: bold; color: #555;">Minimum Headway (one direction)</td><td style="padding: 4px; text-align: right;">{headway}</td></tr>
            <tr style="border-bottom: 1px solid #eee;"><td style="padding: 4px; font-weight: bold; color: #555;">Total Frequency (all directions)</td><td style="padding: 4px; text-align: right;">{freq}</td></tr>
            <tr style="background-color: #f9f9f9;"><td style="padding: 4px; font-weight: bold; color: #555;">Transfer</td><td style="padding: 4px; text-align: right; color: {'green' if transfer == 'true' else 'red'};">{transfer}</td></tr>
        </table>
    </div>
    """

    folium.CircleMarker(
        location=[geom.y, geom.x],
        radius=radius,
        color="#ffffff",
        weight=1.2,
        fill=True,
        fill_color=color,
        fill_opacity=0.9,
        popup=folium.Popup(popup_html, max_width=350),
    ).add_to(hubs_layer)

hubs_layer.add_to(m)

combined_legend_html = """
<div style="
position: fixed;
bottom: 40px;
left: 40px;
z-index: 9999;
background-color: white;
border: 2px solid #666;
border-radius: 8px;
padding: 12px 14px;
font-size: 12px;
line-height: 1.4;
box-shadow: 0 2px 6px rgba(0,0,0,0.25);
max-width: 320px;
font-family: Arial, sans-serif;
">
  <div style="font-weight: bold; font-size: 14px; margin-bottom: 8px;">
    Accessibility Measures
  </div>
  <div style="margin-bottom: 2px;">Base Layer Percentage</div>

  <div style="display:flex; align-items:center; margin:3px 0;"><span style="width:14px;height:14px;background:#ffffb2;display:inline-block;margin-right:8px;border:1px solid #ddd;"></span>less than 2.5%</div>
  <div style="display:flex; align-items:center; margin:3px 0;"><span style="width:14px;height:14px;background:#fed976;display:inline-block;margin-right:8px;border:1px solid #ddd;"></span>2.5% - 5%</div>
  <div style="display:flex; align-items:center; margin:3px 0;"><span style="width:14px;height:14px;background:#feb24c;display:inline-block;margin-right:8px;border:1px solid #ddd;"></span>5% - 10%</div>
  <div style="display:flex; align-items:center; margin:3px 0;"><span style="width:14px;height:14px;background:#fd8d3c;display:inline-block;margin-right:8px;border:1px solid #ddd;"></span>10% - 20%</div>
  <div style="display:flex; align-items:center; margin:3px 0;"><span style="width:14px;height:14px;background:#fc4e2a;display:inline-block;margin-right:8px;border:1px solid #ddd;"></span>20% - 30%</div>
  <div style="display:flex; align-items:center; margin:3px 0;"><span style="width:14px;height:14px;background:#e31a1c;display:inline-block;margin-right:8px;border:1px solid #ddd;"></span>30% - 50%</div>
  <div style="display:flex; align-items:center; margin:3px 0 10px 0;"><span style="width:14px;height:14px;background:#b10026;display:inline-block;margin-right:8px;border:1px solid #ddd;"></span>Greater than 50%</div>

  <div style="border-top:1px solid #ddd; margin:8px 0;"></div>
  <div style="font-weight: bold; margin-bottom: 6px;">Mobility Hubs</div>

  <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#e31700;margin-right:6px;"></span>Cable Car (Trunk)</div>
  <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#ea7164;margin-right:6px;"></span>Cable Car (Branch)</div>
  <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#00491e;margin-right:6px;"></span>Tram/Streetcar/Light Rail (Trunk)</div>
  <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#2ecc71;margin-right:6px;"></span>Tram/Streetcar/Light Rail (Branch)</div>
  <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#0dff00;margin-right:6px;"></span>Rail/Subway/Train (Trunk)</div>
  <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#2980b9;margin-right:6px;"></span>Rail/Subway/Train (Branch)</div>
  <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#f49eff;margin-right:6px;"></span>Bus (Trunk)</div>
  <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#d9d5db;margin-right:6px;"></span>Bus (Branch)</div>
  <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#f1c40f;margin-right:6px;"></span>Bus (Local)</div>
  <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#1f77b4;margin-right:6px;"></span>Ferry (Trunk)</div>
  <div><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#aec7e8;margin-right:6px;"></span>Ferry (Branch)</div>
</div>
"""
m.get_root().html.add_child(folium.Element(combined_legend_html))

folium.LayerControl(collapsed=False).add_to(m)
from folium import Element

# adding custom headers to the layer picker for better organization

section_headers_js = """
<script>
document.addEventListener("DOMContentLoaded", function () {
  function addLayerSectionHeaders() {
    const container = document.querySelector(".leaflet-control-layers-overlays");
    if (!container) return;

    const labels = Array.from(container.querySelectorAll("label"));
    if (!labels.length) return;

    // remove previous injected headers
    container.querySelectorAll(".custom-layer-header").forEach(el => el.remove());

    const accessNames = new Set([
      "Pct Population with Access by Transit",
      "Pct Regional Jobs with Access by Transit",
      "Pct Workers with Access by Transit",
      "Pct Low-Wage Workers with Access by Transit",
      "Pct Low-Med Wage Workers with Access by Transit"
    ]);

    // first access label
    const accessFirst = labels.find(l => accessNames.has(l.textContent.trim()));

    // first non-accessibility overlay AFTER access block
    let infraFirst = null;
    let passedAccess = false;
    for (const l of labels) {
      const t = l.textContent.trim();
      if (accessNames.has(t)) {
        passedAccess = true;
        continue;
      }
      if (passedAccess) {
        infraFirst = l;
        break;
      }
    }

    function makeHeader(text, withTopBorder=true) {
      const hdr = document.createElement("div");
      hdr.className = "custom-layer-header";
      hdr.textContent = text;
      hdr.style.fontWeight = "700";
      hdr.style.fontSize = "12px";
      hdr.style.margin = "8px 0 4px 0";
      hdr.style.paddingTop = "4px";
      if (withTopBorder) hdr.style.borderTop = "1px solid #ddd";
      return hdr;
    }

    if (accessFirst) {
      container.insertBefore(makeHeader("Transit Accessibility Layers", false), accessFirst);
    }

    if (infraFirst) {
      container.insertBefore(makeHeader("Mobility / Infrastructure Layers", true), infraFirst);
    }
  }

  addLayerSectionHeaders();
  setTimeout(addLayerSectionHeaders, 250);
});
</script>
"""
m.get_root().html.add_child(Element(section_headers_js))

from folium import Element


# Collect vector features for export
export_features = []

def add_gdf_to_export(source_gdf, source_name):
    """
    Add a GeoDataFrame to the downloadable GeoJSON export.
    """

    if source_gdf is None or source_gdf.empty:
        return

    export_gdf = source_gdf.copy()

    if export_gdf.crs is None:
        raise ValueError(
            f"The GeoDataFrame for '{source_name}' has no CRS."
        )

    # Convert infinite numeric values to missing values.
    export_gdf = export_gdf.replace(
        [np.inf, -np.inf],
        np.nan
    )

    export_gdf = export_gdf.to_crs(epsg=4326)

    geojson = json.loads(
        export_gdf.to_json(
            na="null",
            drop_id=True
        )
    )

    for feature in geojson.get("features", []):
        feature.setdefault("properties", {})
        feature["properties"]["export_source"] = source_name
        export_features.append(feature)

# Base-layer polygons
add_gdf_to_export(gdf, "base_layer")


# Infrastructure and mobility layers
for obj_name in [
    "BIKE",
    "FTA",
    "AFDC",
    "OSM BIKE PARKING",
    "GBFS",
    "STREETLIGHTS"
]:
    obj = all_objects.get(obj_name)

    if obj is not None and obj.get_is_loaded():
        obj_gdf = obj.gdf

        if obj_gdf is not None and not obj_gdf.empty:
            add_gdf_to_export(obj_gdf, obj_name)


# Mobility hubs
if mobility_hub_instance.get_is_loaded():
    mobility_hub_gdf = mobility_hub_instance.gdf

    if mobility_hub_gdf is not None and not mobility_hub_gdf.empty:
        add_gdf_to_export(
            mobility_hub_gdf,
            "MOBILITY_HUBS"
        )


export_geojson = {
    "type": "FeatureCollection",
    "features": export_features,
}

# Serialize safely for JavaScript.
# The replacement prevents a </script> sequence from prematurely
# closing the script element.
export_geojson_json = json.dumps(
    export_geojson,
    allow_nan=False
).replace("</", "<\\/")


map_name = m.get_name()

geojson_export_control = f"""
<script>
document.addEventListener("DOMContentLoaded", function () {{
    const map = {map_name};

    const exportGeoJSON = {export_geojson_json};

    const ExportGeoJSONControl = L.Control.extend({{
        options: {{
            position: "topleft"
        }},

        onAdd: function (map) {{
            const container = L.DomUtil.create(
                "div",
                "leaflet-bar leaflet-control"
            );

            const button = L.DomUtil.create(
                "button",
                "",
                container
            );

            button.type = "button";
            button.title = "Export map data as GeoJSON";
            button.setAttribute(
                "aria-label",
                "Export map data as GeoJSON"
            );
            button.innerHTML = "⬇️";

            button.style.backgroundColor = "white";
            button.style.border = "none";
            button.style.cursor = "pointer";
            button.style.width = "30px";
            button.style.height = "30px";
            button.style.fontSize = "16px";
            button.style.lineHeight = "28px";
            button.style.padding = "0";

            button.addEventListener("mouseenter", function () {{
                button.style.backgroundColor = "#f4f4f4";
            }});

            button.addEventListener("mouseleave", function () {{
                button.style.backgroundColor = "white";
            }});

            L.DomEvent.disableClickPropagation(container);
            L.DomEvent.disableScrollPropagation(container);

            L.DomEvent.on(button, "click", function () {{
                const blob = new Blob(
                    [JSON.stringify(exportGeoJSON, null, 2)],
                    {{ type: "application/geo+json;charset=utf-8" }}
                );

                const url = URL.createObjectURL(blob);
                const link = document.createElement("a");

                link.href = url;
                link.download = "map_data.geojson";
                link.style.display = "none";

                document.body.appendChild(link);
                link.click();
                document.body.removeChild(link);

                setTimeout(function () {{
                    URL.revokeObjectURL(url);
                }}, 100);
            }});

            return container;
        }}
    }});

    map.addControl(new ExportGeoJSONControl());
}});
</script>
"""

m.get_root().html.add_child(Element(geojson_export_control))

m.save("display_map_final_exportable.html")
m

### Example Export Function

In [ ]:
import pathlib
def export_data_object_as_layer(data_object_instance, output_folder):
    """Export data_object_instance to output_folder as a geojson"""
    gdf = data_object_instance.gdf
    gdf.to_file(pathlib.Path(output_folder) / f"{data_object_instance.name}.geojson")

objects_to_export = [gtfs_instance, mobility_hub_instance, gbfs_instance, base_layer_instance, afdc_instance, fta_instance]
for object in objects_to_export:
    print(object.name)
    export_data_object_as_layer(object, "test")
